In [14]:
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import skew
import pandas as pd

class DataGeneration():
    def __init__(self, mus=[], sigmas=[], rho=[] , w_portfolio=[], asset_names=None):
        self.mus    = np.array(mus) 
        self.sigmas = np.array(sigmas) 
        self.rho    = np.array(rho)
        self.w_port = np.array(w_portfolio)
        D = np.diag(self.sigmas)
        self.covariance = D @ self.rho @ D 
        
        n_assets = self.mus.shape[0]
        self.asset_names =  (asset_names if asset_names is not None \
                            else [f"Asset {i}" for i in range(n_assets)]
        )
        self.Rt = None
    
    def generate(self, T, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.Rt =  np.random.multivariate_normal( self.mus, self.covariance, size=T)
        return self.Rt 
    
    def plot(self):
        if self.Rt is None: 
            raise RuntimeError("Generate returns before calling plot()")
        plt.hist(self.Rt)
        plt.show()
    
    def summary(self):
        if self.Rt is None: 
            raise RuntimeError("Generate returns before calling summary()")
        #quantiles
        q_low, q_high  = np.percentile(self.Rt, [15.9, 84.1], axis=0)

        results = {
            "mean":   np.mean(self.Rt, axis=0),
            "std":    np.std(self.Rt, axis=0),
            "median": np.median(self.Rt, axis=0),
            "q15.9":  q_low,
            "q84.1":  q_high,
            "skew":   skew(self.Rt, axis=0),
        }
        correlations = np.corrcoef(self.Rt.T)
        df_stats = pd.DataFrame(results, index=self.asset_names)
        df_corr  = pd.DataFrame( correlations , index=self.asset_names, columns=self.asset_names)
        return df_stats, df_corr


mus = [0.10, 0.07, 0.04]
sigmas = [0.25, 0.15, 0.07]
rho = [
    [1.0, 0.6, 0.2],
    [0.6, 1.0, 0.3],
    [0.2, 0.3, 1.0]
]

gen = DataGeneration(mus, sigmas, rho)
returns = gen.generate(T=10_000, seed=42)
results_stat, results_corr = gen.summary()
results_corr
#gen.plot()


,Asset 0,Asset 1,Asset 2
Asset 0,1.000000,0.598403,0.216395
Asset 1,0.598403,1.000000,0.311701
Asset 2,0.216395,0.311701,1.000000
